In [1]:
import requests
import pandas as pd

# 1. CONFIGURACIONES GLOBALES (Tus datos fijos)
# ¡ATENCIÓN! URL actualizada a PRODUCCIÓN
BASE_URL = "https://data-api-f7y55nboxq-uc.a.run.app"
CLIENT_ID = "abastecimiento_app"
CLIENT_SECRET = "0QPwdt6rPjnlUuRdpVYKeCqy6qOxVZv5"

# --- FUNCIÓN 1: OBTENER TOKEN ---
def obtener_token(client_id, client_secret, base_url):
    """Va a la API, se identifica y devuelve el pase VIP temporal."""
    print("Obteniendo pase VIP (Token)...")
    auth_url = f"{base_url}/auth/token"
    auth_data = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret
    }
    
    response = requests.post(auth_url, data=auth_data)
    
    if response.status_code == 200:
        print("¡Token obtenido con éxito!\n")
        return response.json().get("access_token")
    else:
        # Si falla la contraseña, detenemos todo el programa con un error
        raise Exception(f"Error de autenticación: {response.text}")


# --- FUNCIÓN 2: DESCARGAR CUALQUIER TABLA ---
def descargar_tabla(base_url, endpoint, token):
    """Descarga automáticamente todas las páginas de la tabla que le pidas."""
    print(f"Descargando tabla completa de '/{endpoint}'...")
    headers = {"Authorization": f"Bearer {token}"}
    
    # Armamos la URL inicial forzando la carga de 100 en 100
    siguiente_pagina = f"{base_url}/{endpoint}?limit=100" 
    todas_las_filas = []

    # El bucle de paginación que ya conoces
    while siguiente_pagina:
        response = requests.get(siguiente_pagina, headers=headers).json()
        todas_las_filas.extend(response.get("data", []))
        siguiente_pagina = response.get("next_url")

    # Convertimos a Pandas y devolvemos el resultado
    df = pd.DataFrame(todas_las_filas)
    print(f"¡Listo! Tabla '{endpoint}' descargada con {len(df)} filas.\n")
    return df


# --- EJECUCIÓN PRINCIPAL ---
# Aquí es donde realmente mandamos a ejecutar las "recetas" que creamos arriba
if __name__ == "__main__":
    try:
        # Paso A: Nos autenticamos UNA SOLA VEZ
        mi_token = obtener_token(CLIENT_ID, CLIENT_SECRET, BASE_URL)
        
        # Paso B: Descargamos la tabla 1
        df_master = descargar_tabla(BASE_URL, "planilla-master", mi_token)
        
        # Paso C: Descargamos la tabla 2 usando EL MISMO TOKEN
        df_panel = descargar_tabla(BASE_URL, "panel-ci", mi_token)
        
        # Verificamos qué tenemos
        print("---------- RESUMEN FINAL ----------")
        print(f"Planilla Master guardada en 'df_master' -> Dimensiones: {df_master.shape}")
        print(f"Panel C&I guardada en 'df_panel'        -> Dimensiones: {df_panel.shape}")
        
    except Exception as error:
        print("El proceso falló:", error)

Obteniendo pase VIP (Token)...
¡Token obtenido con éxito!

Descargando tabla completa de '/planilla-master'...
¡Listo! Tabla 'planilla-master' descargada con 1752 filas.

Descargando tabla completa de '/panel-ci'...
¡Listo! Tabla 'panel-ci' descargada con 319 filas.

---------- RESUMEN FINAL ----------
Planilla Master guardada en 'df_master' -> Dimensiones: (1752, 228)
Panel C&I guardada en 'df_panel'        -> Dimensiones: (319, 166)


In [1]:
df_master.fecha_de_inicio_de_instalacion_real
df_master.fecha_de_termino_de_instalacion_real
df_master.fecha_de_ganado
df_master.kwh

NameError: name 'df_master' is not defined

In [3]:
# Filtra la lista iterando columna por columna
columnas_fecha = [col for col in df_master.columns if 'fecha' in col.lower()]

print(columnas_fecha)

['fecha_pago_10', 'fecha_pago_30', 'fecha_pago_60', 'fecha_reserva', 'fecha_de_ganado', 'fecha_cotizacion', 'fecha_ingreso_f3', 'fecha_nacimiento', 'fecha_pago_status', 'fecha_prorroga_f4', 'fecha_respuesta_f4', 'fecha_te4_inscrito', 'fecha_te4_declarado', 'fecha_te4_rechazado', 'fecha_3_meses_de_uso', 'fecha_compra_equipos', 'fecha_inicio_informe', 'fecha_reserva_status', 'fecha_traspaso_venta', 'fecha_vencimiento_f4', 'fecha_visita_tecnica', 'fecha_de_carga_boleta', 'fecha_inicio_pausa_n1', 'fecha_inicio_pausa_n2', 'fecha_cancelado_status', 'fecha_conectado_status', 'fecha_reingreso_de_te4', 'fecha_termino_garantia', 'fecha_termino_pausa_n1', 'fecha_termino_pausa_n2', 'fecha_de_carga_contrato', 'fecha_de_visita_tecnica', 'fecha_validacion_status', 'fecha_2da_visita_tecnica', 'fecha_cierre_de_contrato', 'fecha_instalacion_status', 'fecha_ok_para_certificar', 'fecha_entrega_del_reporte', 'fecha_actualizacion_avance', 'fecha_certificacion_status', 'fecha_de_carga_de_anticipo', 'fecha_de

In [4]:
df_panel.columns

Index(['dn', 'dx', 'ceco', 'oocc', 'venta', 'comuna', 'region', 'status',
       'anticipo', 'contador',
       ...
       'avance_seleccion_y_adjudicacion_de_subcontrato',
       'fin_proceso_de_homologacion_de_of_carta_de_adju',
       'fin_proceso_de_homologacion_de_oferta_carta_de_a',
       'inicio_proceso_de_homologacion_de_oferta_carta_d',
       'semaforo_proceso_de_homologacion_de_oferta_carta',
       'semaforo_seleccion_y_adjudicacion_de_subcontrato',
       'fin_seleccion_y_adjudicacion_de_subcontrato_gantt',
       'fin_seleccion_y_adjudicacion_de_subcontrato_planif',
       'inicio_seleccion_y_adjudicacion_de_subcontrato_gan',
       'inicio_seleccion_y_adjudicacion_de_subcontrato_pla'],
      dtype='object', length=166)